### json 파일 로드 
- TL_text_entailment.json 파일을 로드 
- context(원문) 데이터와 question(질문) 데이터, answers에 있는 text 데이터들을 이용하여 하나의 데이터프레임으로 변환

In [ ]:
import json 
import pandas as pd 
import torch 
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments, 
    DataCollatorForSeq2Seq
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from pprint import pprint

In [ ]:
# 데이터 로드 
with open("./data/TL_text_entailment.json", 'r', encoding='utf-8') as f:
    raw = json.load(f)

pprint(raw)

In [ ]:
data = []
for doc in raw['data']:
    for para in doc['paragraphs']:
        if len(para) >= 2:
            print(para['context'])
        context = para['context']
        for qa in para['qas']:
            question = qa['question']
            answer = qa['answers']['text']
            data.append({
                'context' : context, 
                'question' : question, 
                'answer' : answer
            })            
df = pd.DataFrame(data)

In [ ]:
df.info()

In [ ]:
df['answer'].value_counts()

In [ ]:
# 초기 설정 값
model_name = 'digit82/kobart-summarization'

# 입력(원문 + 질문)의 최대 토큰의 개수 
max_input_len = 512
# 출력의 최대 토큰의 개수 (예 / 아니요) -> 짧아도 가능 
max_target_len = 8

# 원본의 정답을 숫자형으로 변경 
ans2id = {'No' : 0, 'Yes': 1}
# 숫자형 정답형 문자로 변경 dict
id2target = {0 : '아니오', 1 : '예'}

In [ ]:
# 고정 문구가 있는 원문 + 질문 데이터를 생성하는 함수 
def build_prompt(context, question):
    return (
        f"다음 글을 읽고 문장이 참이면 '예', 거짓이면 '아니오'로 답하시오\n"
        f"원문 : {context}\n"
        f"문장 : {question}"
    )

In [ ]:
inputs = df[['context', 'question']].apply(lambda x : build_prompt(*(x.tolist())), axis=1).tolist()

In [ ]:
inputs

In [ ]:
# answer의 데이터는 예 아니오 로 변경 
targets = df['answer'].map(ans2id).map(id2target).tolist()

In [ ]:
ds = Dataset.from_dict(
    {
        'input_text' : inputs, 
        'target_text' : targets
    }
)

In [ ]:
ds

In [ ]:
# 데이터 분할시 사용할 target 데이터는 현재 정수형 데이터셋 
# 계층화 분할에서는 ClassLabel 타입만 가능 (Dataset에 내장된 train_test_split)
# strat 컬럼을 생성하는 이유는 계층화에서 사용하고 제외시킬 컬럼
ds = ds.map(lambda x : {'strat' : 0 if x['target_text'] == '아니오' else 1})
ds

In [ ]:
# strat 컬럼의 데이터를 ClassLabel 타입으로 변경 
ds = ds.class_encode_column('strat')

In [ ]:
#  Dataset 에서 train_test_split() 함수 사용 
split1 = ds.train_test_split(
    test_size=0.2, seed = 42, stratify_by_column='strat'
)
split1

In [ ]:
train_ds = split1['train'].remove_columns('strat')

In [ ]:
train_ds['target_text']

In [ ]:
split2 = split1['test'].train_test_split(
    test_size=0.5, seed = 42, stratify_by_column='strat'
)

In [ ]:
valid_ds = split2['train'].remove_columns('strat')
test_ds = split2['test'].remove_columns('strat')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def token_fn(batch):
    inputs = tokenizer(
        batch['input_text'], 
        max_length = max_input_len, 
        truncation = True, 
        return_token_type_ids = False
    )
    labels = tokenizer(
        batch['target_text'], 
        max_length = max_target_len, 
        truncation = True, 
        return_token_type_ids = False
    )
    inputs['labels'] = labels['input_ids']
    return inputs

In [ ]:
cols = ['input_text', 'target_text']
train_ds = train_ds.map(token_fn, batched=True, remove_columns=cols)
valid_ds = valid_ds.map(token_fn, batched=True, remove_columns=cols)
test_ds = test_ds.map(token_fn, batched=True, remove_columns=cols)

In [ ]:
len(valid_ds['labels'])

In [ ]:
# 모델 생성 
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    num_labels = 2,                 # num_labels = 3을 덮어씌워준다. 
    id2label = {0 : "LABEL_0", 1 : 'LABEL_1'}, 
    label2id = {"LABEL_0" : 0, "LABEL_1" : 1}
)

In [ ]:
# 패딩 토큰 동적으로 배치 
data_collator = DataCollatorForSeq2Seq(
    tokenizer= tokenizer, 
    model = model, 
    label_pad_token_id= -100
)

In [ ]:
import numpy as np 

In [ ]:
# 평가 지표 생성  -> 예측 값이 예 / 아니오  --> 0 / 1 재 변환 
def text_to_id(text):
    # 예측으로 생성된 텍스트 :   예 예 예 예 , 아니오 니오 오오오오오
    # 만약에 문자의 시작이 예로 시작한다면? 
    t = text.strip()
    if t.startswith("예"):
        result = 1
    elif t.startswith('아니'):
        result = 0
    else:
        result = -1
    return result

def metrics(eval_pred):
    preds, labels = eval_pred

    # preds 값이 튜플의 형태로 들어오는 경우 -> 생성된 토큰의 개수가 1개인 경우 
    if isinstance(preds, tuple):
        preds = preds[0]
    # 3차원 데이터인 경우 2차원으로 데이터를 줄이겠다. 
    if len(preds.shape) == 3:
        preds = np.argmax(preds, axis=-1)
        
    # 패딩 토큰의 인코딩 값들을 tokenizer의 패딩 토큰 아이디로 변경 
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )

    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 문자열 데이터를 0/1의 형태로 변경 
    pred_ids = [text_to_id(t) for t in pred_str]
    label_ids = [text_to_id(t) for t in label_str]

    acc = accuracy_score(pred_ids, label_ids)
    f1 = f1_score(pred_ids, label_ids, average='macro')

    fail_rate = np.mean( [p == -1 for p in pred_ids] )
    return {
        'accuracy' : round(acc * 100 , 2), 
        'f1' : f1, 
        'parse_fail_rate' : fail_rate
    }


In [ ]:
# trainer 설정 값 세팅
args = Seq2SeqTrainingArguments(
    output_dir="./kobart", 
    num_train_epochs=3, 
    learning_rate= 3e-05, 
    weight_decay=0.01, 
    warmup_ratio=0.1, 
    eval_strategy='epoch', 
    save_strategy='epoch', 
    load_best_model_at_end=True, 
    metric_for_best_model='f1', 
    greater_is_better=True, 
    generation_max_length=max_target_len, 
    logging_steps=1, 
    per_device_train_batch_size=16,         # 메모리 부족 현상 나타나면 8로 변경
    per_device_eval_batch_size= 32
)
trainer = Seq2SeqTrainer(
    model = model, 
    args = args,
    train_dataset= train_ds.select(range(500)), 
    eval_dataset = valid_ds.select(range(100)), 
    processing_class= tokenizer, 
    data_collator= data_collator, 
    compute_metrics= metrics
)

In [ ]:
trainer.train()

In [ ]:
model2 = AutoModelForSeq2SeqLM.from_pretrained("./best")

In [ ]:
# 데이터 추론 -> 학습 된 모델에서 예측 
def predict(context, question):
    # context : 원문 
    # question : 질문(가설)

    # 학습 때와 같은 형태의 프롬프트를 생성 
    prompt = build_prompt(context, question)
    inputs = tokenizer(
        prompt, 
        max_length = max_input_len, 
        truncation = True, 
        return_tensors = 'pt', 
        return_token_type_ids = False
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs, 
            max_new_tokens = max_target_len, 
            num_beams = 1
        )
    text = tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

    pred_id = text_to_id(text)      # 0 / 1 / -1 변환
    label = {0 : '아니오(거짓)', 1 : '예(참)', -1 : '측정불가'}[pred_id]

    return label, text

In [ ]:
context_test = (
    "최고위원 선거에서는 민주당 초선모임 '처럼회' 맴버로 중대범죄수사청 설치 등을" 
    "주장해 온 강성 친문 김용민 의원이 가장 많은 표를 얻었다."
)
question_test = "처럼회는 처음으로 국회의원 직무를 수행하는 민주당 의원들의 모임이야"

label, raw = predict(context_test, question_test)

print("생성된 텍스트 : ", raw, "판정 : ", label)

In [ ]:
# test_ds 데이터셋에서 예측과 정답이 일치하는가? 확인하려면?
# 상위 5개를 데이터의 예측값과 실젯값을 출력 

